# HTP 전용 객체탐지 모델 파인튜닝 (Stage B)

**산출물 2: HTP 47클래스 전용 탐지 모델.** 범용 손그림 모델을 시작 가중치로
HTP 데이터셋에 파인튜닝해 `htp_yolo/<HTP_MODEL_NAME>` 을 만든다.

`baseline.ipynb` 비교 실험에서 **손그림 사전학습 → HTP**가 **COCO → HTP** 직접 파인튜닝보다
유효함을 확인했다. 이 노트북은 그 결론을 전제로 대조군 없이 sketch → HTP 경로만 학습한다.

### 선행 조건
시작 가중치 `.pt` 파일이 **먼저 있어야 한다.** 없으면 Config 셀에서 즉시 중단된다.
[sketch_base.ipynb](sketch_base.ipynb)를 먼저 실행하거나,
`model_download.ipynb`로 기존 가중치를 내려받는다.

```
sketch_base.ipynb ──> sketch_base*.pt ──> (이 노트북) ──> htp_best*.pt
```

의존은 **파일 경유 단방향**이다. 이 노트북은 그 `.pt`를 누가 만들었는지, 클래스가 몇 종인지
알 필요가 없다 — 클래스 수가 달라도 **검출 헤드는 자동 재초기화**되고 백본/넥의 선화 특징만 전이된다.

### ⚠️ 캐시 무효화 규칙
| 바꾼 것 | 지워야 반영되는 것 |
|---|---|
| `SUBSET_PER_TYPE`, `TEST_FRACTION` | `htp_yolo/stageB_htp/` (바꾸면 반드시 삭제) |
| 학습값(`HTP_EPOCHS`, `HTP_TRAIN_KW` …) | 없음 — 매번 재학습된다 |

### ⚠️ 이 노트북은 실행할 때마다 재학습한다
`HTP_MODEL_NAME` 파일을 매번 덮어쓴다. 시작 가중치를 바꿔 실험할 때는
`HTP_MODEL_NAME`과 `RUN_NAME`을 함께 바꿔 기존 산출물을 보존할 것.

Config를 고친 뒤에는 **커널 재시작 → Run All**.

## 1. 환경 세팅

In [ ]:
# 필요 패키지 설치 (JupyterHub/VSCode 공통). 이미 설치돼 있으면 그대로 통과.
%pip install -q "ultralytics>=8.3.0" opencv-python-headless pillow pyyaml tqdm matplotlib pandas

In [ ]:
# Stage B는 HTP zip 접근이 필요하다 — Quick Draw 관련 의존은 없다.
import os

# ── GPU 선택: 물리 GPU 6번만 사용 (공용 서버 지정) ────────────────
#   ⚠️ import torch 보다 반드시 먼저 실행돼야 적용된다. 이미 torch가 로드된
#      뒤면 무시되므로 커널 재시작 후 이 셀부터 다시 실행할 것.
#   ⚠️ 값은 문자열이어야 한다. CUDA_VISIBLE_DEVICES="6" 을 걸면 torch에는
#      그 GPU가 유일한 장치(cuda:0)로 보이므로 Config의 DEVICE 는 6이 아니라
#      0 으로 두어야 한다(그대로 두면 됨).
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

import sys, json, random, shutil, math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm.auto import tqdm

try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    TORCH_VER = torch.__version__
except ModuleNotFoundError:
    torch = None; CUDA_AVAILABLE = False; TORCH_VER = "(미설치)"

print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("PyTorch     :", TORCH_VER)
print("CUDA 사용가능:", CUDA_AVAILABLE)
if CUDA_AVAILABLE:
    print("GPU         :", torch.cuda.get_device_name(0), f"(x{torch.cuda.device_count()})")

In [ ]:
import io, zipfile
from PIL import Image

In [ ]:
# ── 한글 폰트 설정 (matplotlib 라벨 깨짐 방지) ──────────────────────
# Windows/Linux/macOS 공통. 이 셀을 먼저 실행하면 이후 모든 시각화에 한글이 적용된다.
# JupyterHub(리눅스, sudo 없음)처럼 시스템에 한글 폰트가 없으면 NanumGothic을
# 유저 폴더에 자동 내려받아 등록한다 — apt/root 권한이 필요 없다.
import urllib.request
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 유저 홈에 폰트를 캐시(쓰기 가능·재실행 시 재사용). 실패하면 현재 폴더로 대체.
_FONT_URL = ("https://github.com/google/fonts/raw/main/ofl/nanumgothic/"
             "NanumGothic-Regular.ttf")
try:
    _FONT_CACHE = Path.home() / ".fonts"
    _FONT_CACHE.mkdir(parents=True, exist_ok=True)
except Exception:
    _FONT_CACHE = Path("fonts"); _FONT_CACHE.mkdir(exist_ok=True)
_FONT_DL = _FONT_CACHE / "NanumGothic-Regular.ttf"


def _download_korean_font():
    # 이미 받아둔 게 있으면 재사용, 없으면 내려받는다. 반환: 성공 시 파일 경로.
    if _FONT_DL.exists() and _FONT_DL.stat().st_size > 100_000:
        return _FONT_DL
    try:
        req = urllib.request.Request(_FONT_URL, headers={"User-Agent": "htp/1.0"})
        with urllib.request.urlopen(req, timeout=120) as r:
            _FONT_DL.write_bytes(r.read())
        print(f"NanumGothic 다운로드: {_FONT_DL}")
        return _FONT_DL
    except Exception as e:
        print(f"⚠️ 폰트 다운로드 실패({type(e).__name__}: {e}) — 네트워크/프록시를 확인하세요.")
        return None


def _apply_font(fp):
    font_manager.fontManager.addfont(str(fp))
    name = font_manager.FontProperties(fname=str(fp)).get_name()
    plt.rcParams["font.family"] = name
    plt.rcParams["axes.unicode_minus"] = False
    return name


def setup_korean_font():
    candidates = [
        r"C:\Windows\Fonts\malgun.ttf",        # Windows 맑은 고딕
        r"C:\Windows\Fonts\NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",          # Linux (apt)
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/System/Library/Fonts/AppleSDGothicNeo.ttc",               # macOS
        "/Library/Fonts/AppleGothic.ttf",
        str(_FONT_DL),                          # 이전 실행에서 받아둔 것
    ]
    for fp in candidates:
        if Path(fp).exists():
            name = _apply_font(fp)
            print(f"한글 폰트: {name}  ({fp})")
            return fp, name
    # 설치된 폰트 이름으로 시도(파일 경로로 못 찾은 경우)
    for name in ["Malgun Gothic", "NanumGothic", "AppleGothic", "Noto Sans CJK KR"]:
        if any(f.name == name for f in font_manager.fontManager.ttflist):
            plt.rcParams["font.family"] = name
            plt.rcParams["axes.unicode_minus"] = False
            print(f"한글 폰트(이름): {name}")
            return None, name
    # 마지막 수단: 유저 폴더로 다운로드해 등록 (JupyterHub 등 root 없는 환경)
    fp = _download_korean_font()
    if fp is not None:
        name = _apply_font(fp)
        print(f"한글 폰트(다운로드): {name}  ({fp})")
        return str(fp), name
    print("⚠️ 한글 폰트를 찾지 못했습니다. 라벨이 깨질 수 있습니다.")
    return None, None

KOREAN_FONT_PATH, KOREAN_FONT_NAME = setup_korean_font()

## 2. Config  ⚙️

**환경을 바꿀 때는 이 셀만 수정한다.** 기본값은 스모크 테스트용이다.

In [ ]:
# ═══════════════════════ Config (Stage B) ═══════════════════════
# 폴더/zip 이름이 바뀌어도 동작하도록 "탐색"으로 경로를 잡는다.
# (한글/영문 명명 혼재, 폴더명 변경에 견디게 하기 위함)
def _find_data_root():
    here = Path.cwd()
    for base in [here, *here.parents]:
        for cand in (base / "data", base / "ai" / "models" / "data"):
            if cand.exists():
                return cand
    return here / "data"

def _find_htp_root(data_root: Path):
    # Training/ 과 Validation/ 을 모두 가진 폴더를 하위 2단계까지 탐색
    cands = [data_root]
    if data_root.exists():
        for p in data_root.iterdir():
            if p.is_dir():
                cands.append(p)
                cands.extend(q for q in p.iterdir() if q.is_dir())
    for c in cands:
        if (c / "Training").is_dir() and (c / "Validation").is_dir():
            return c
    return None

def _find_subdir(split_dir: Path, kind: str):
    # kind='src'(원천/이미지) | 'lbl'(라벨). 한글/영문 폴더명 모두 인식
    keys = ("원천", "source", "image", "img") if kind == "src" else ("라벨", "label")
    for p in sorted(split_dir.iterdir()):
        if p.is_dir() and any(k in p.name.lower() for k in keys):
            return p
    return None

def _zip_type(stem: str):
    # 파일명에서 그림 유형 판정.
    # 주의: 'woman'이 'man'을 포함하므로 여자사람을 먼저 검사해야 한다.
    s = stem.lower()
    if "여자사람" in s or "woman" in s: return "여자사람"
    if "남자사람" in s or "man"   in s: return "남자사람"
    if "나무"     in s or "tree"  in s: return "나무"
    if "집"       in s or "house" in s: return "집"
    return None

DATA_ROOT   = _find_data_root()
HTP_ROOT    = _find_htp_root(DATA_ROOT)
# ⚠️ 절대경로 필수: 상대경로를 쓰면 Ultralytics가 project= 인자를
#    ~/AppData/Roaming/Ultralytics/settings.json 의 runs_dir(다른 프로젝트 설정값일 수 있음)와
#    합쳐 엉뚱한 위치(runs/detect/htp_yolo/runs/...)에 저장해버린다. (실제 발생한 함정)
OUTPUT_ROOT = Path("htp_yolo").resolve()         # 산출물 루트 (sketch_base.ipynb와 공유)
HTP_DS_DIR  = OUTPUT_ROOT / "stageB_htp"         # HTP YOLO 데이터셋
RUNS_DIR    = OUTPUT_ROOT / "runs"

HTP_TYPES = ["집", "나무", "남자사람", "여자사람"]

# ---- 시작 가중치 (sketch_base.ipynb 산출물) ----
#   확장판으로 실험하려면 여기를 "sketch_base_v2.pt"로 바꾸고,
#   아래 HTP_MODEL_NAME·RUN_NAME도 함께 바꿔 기존 산출물을 보존한다.
SKETCH_MODEL_NAME = "sketch_base.pt"
SKETCH_BASE       = OUTPUT_ROOT / SKETCH_MODEL_NAME

# ---- 산출물 ----
HTP_MODEL_NAME = "htp_best.pt"
HTP_BEST       = OUTPUT_ROOT / HTP_MODEL_NAME
RUN_NAME       = "htp_from_sketch"    # 학습 로그 폴더명

# ---- 학습 하이퍼파라미터 ----
MODEL_ARCH = "yolo11s.pt"   # 기록용 — 실제 아키텍처는 SKETCH_BASE의 것을 따른다
IMG_SIZE   = 960            # 스모크: 640 / 본학습: 960~1280 (원본 1280)
BATCH      = 16
WORKERS    = 8
DEVICE     = 0 if CUDA_AVAILABLE else "cpu"   # 오버라이드: "cpu" / 0 / "0,1"
SEED       = 42

HTP_EPOCHS      = 20      # 스모크: 5 / 본학습: 50~100
PATIENCE        = 20
SUBSET_PER_TYPE = 1000    # 스모크: 유형별 200장. None이면 전체(유형별 11,200장)
TEST_FRACTION   = 0.10    # 공식 Training에서 test로 분리할 비율 (val은 공식 Validation 사용)

# ---- 추가 튜닝: Ultralytics train() 인자를 자유롭게 전달 ----
# 예) {"lr0": 0.001, "cos_lr": True, "mosaic": 0.5, "optimizer": "AdamW", "close_mosaic": 10}
HTP_TRAIN_KW = {"cos_lr": True, "freeze": 10}

# 데이터 소스 인덱싱(폴더/zip 자동 판별)은 아래 '데이터 접근' 셀에서 수행한다.

random.seed(SEED); np.random.seed(SEED)
for d in (OUTPUT_ROOT, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT    :", DATA_ROOT)
print("HTP_ROOT     :", HTP_ROOT)
if HTP_ROOT is None:
    print("  ⚠️ Training/Validation 폴더를 찾지 못했습니다. DATA_ROOT를 직접 지정하세요.")
else:
    print("  데이터 소스 판별·유형별 확보 현황은 아래 '데이터 접근' 셀에서 출력됩니다.")
print("OUTPUT_ROOT  :", OUTPUT_ROOT)
print("시작 가중치  :", SKETCH_BASE)
print("산출물       :", HTP_BEST, "(덮어씀)" if HTP_BEST.exists() else "(신규)")
print(f"imgsz={IMG_SIZE} batch={BATCH} device={DEVICE} epochs={HTP_EPOCHS}")
print(f"SUBSET_PER_TYPE={SUBSET_PER_TYPE} (None=전체) · HTP_TRAIN_KW={HTP_TRAIN_KW}")

# 시작 가중치는 학습 직전이 아니라 '여기'에서 검사한다 — 데이터 변환에 수 분을 쓰고 나서
# 없는 걸 알게 되면 그 시간이 통째로 낭비된다.
if not SKETCH_BASE.exists():
    raise FileNotFoundError(
        f"시작 가중치가 없습니다: {SKETCH_BASE}\n"
        "→ sketch_base.ipynb를 먼저 실행하거나 model_download.ipynb로 내려받으세요.\n"
        "   (COCO에서 바로 파인튜닝하는 것도 가능하지만 성능이 열세인 것이 baseline.ipynb에서 확인됨)"
    )
print("\n✓ 시작 가중치 확인 완료")

## 3. 데이터 탐색 (HTP)

### 파악된 HTP 어노테이션 포맷 (JSON)
```
{
  "meta": { "img_resolution": "1280x1280", "age": 10, "sex": "남", ... },
  "annotations": {
    "class": "집",
    "bbox": [ {"label": "집전체", "x": 220, "y": 222, "w": 402, "h": 529}, ... ]
  }
}
```
- `x, y` = bbox **좌상단** 픽셀 좌표, `w, h` = 픽셀 폭/높이. 모든 이미지 **1280×1280**.
- 라벨 파일명 = 이미지 파일명(확장자만 다름): `집_10_남_00137.json` ↔ `집_10_남_00137.jpg`.
- 검증 결과: 이미지 50,400장 / 라벨 50,400개 **완전 1:1 매칭, 미매칭 0**.

In [ ]:
# ── HTP 데이터 접근 (폴더/zip 양쪽 지원) ───────────────────────────
#   레이아웃 두 가지를 자동 판별한다:
#     (A) 폴더:  HTP_ROOT/<split>/<유형>/*.jpg,*.json   (압축 해제됨 — JupyterHub 서버)
#     (B) zip :  HTP_ROOT/<split>/(원천|라벨)/*.zip      (AI-Hub 원본 — 로컬)
#   어느 쪽이든 HTP_INDEX[split][htype] = {"img":{stem:ref}, "lbl":{stem:ref}} 로 통일한다.
#   ref: 폴더모드=Path, zip모드=(zip_path, 멤버명). 실제 읽기는 htp_read_* 가 흡수한다.
def fix_name(name: str) -> str:
    # zip 내부 파일명 유니코드 복원(cp437로 깨진 경우만 보정). 폴더 경로엔 영향 없음.
    if "\ufffd" in name:
        try:
            return name.encode("cp437").decode("utf-8")
        except Exception:
            pass
    return name

def _zip_names(zip_path, ext):
    with zipfile.ZipFile(zip_path) as z:
        return [n for n in z.namelist() if n.lower().endswith(ext)]

def _index_split(split_dir):
    # 유형별 {"img":{stem:ref}, "lbl":{stem:ref}} 반환. 폴더 우선, 없으면 zip.
    result = {}
    # (A) 폴더 모드: <split>/<유형>/ 안의 jpg·json 직접 인덱싱
    for htype in HTP_TYPES:
        tdir = split_dir / htype
        if tdir.is_dir() and next(tdir.rglob("*.jpg"), None) is not None:
            result[htype] = {
                "img": {p.stem: p for p in tdir.rglob("*.jpg")},
                "lbl": {p.stem: p for p in tdir.rglob("*.json")},
            }
    if result:
        return result
    # (B) zip 모드: 원천/라벨 zip에서 멤버 인덱싱
    src_zips, lbl_zips = {}, {}
    for _kind, _reg in (("src", src_zips), ("lbl", lbl_zips)):
        sub = _find_subdir(split_dir, _kind)
        if sub is not None:
            for z in sorted(sub.glob("*.zip")):
                t = _zip_type(z.stem)
                if t:
                    _reg[t] = z
    for htype in HTP_TYPES:
        entry = {"img": {}, "lbl": {}}
        if htype in src_zips:
            zp = src_zips[htype]
            entry["img"] = {Path(fix_name(n)).stem: (zp, n) for n in _zip_names(zp, ".jpg")}
        if htype in lbl_zips:
            zp = lbl_zips[htype]
            entry["lbl"] = {Path(fix_name(n)).stem: (zp, n) for n in _zip_names(zp, ".json")}
        if entry["img"] or entry["lbl"]:
            result[htype] = entry
    return result

HTP_INDEX = {}
if HTP_ROOT is not None:
    for _split in ("Training", "Validation"):
        HTP_INDEX[_split] = _index_split(HTP_ROOT / _split)

def htp_members(split, htype, kind):
    # kind: "img" | "lbl"  →  {stem: ref}
    return HTP_INDEX.get(split, {}).get(htype, {}).get(kind, {})

def htp_read_bytes(ref) -> bytes:
    if isinstance(ref, tuple):                 # zip 모드: (zip_path, member)
        with zipfile.ZipFile(ref[0]) as z:
            return z.read(ref[1])
    return Path(ref).read_bytes()              # 폴더 모드: 파일 경로

def htp_read_json(ref) -> dict:
    return json.loads(htp_read_bytes(ref).decode("utf-8"))

# 유형별 확보 현황(이미지 수) + 누락 경고
_ok = True
for _s in ("Training", "Validation"):
    _cnts = {t: len(htp_members(_s, t, "img")) for t in HTP_TYPES}
    _miss = [t for t, n in _cnts.items() if n == 0]
    if _miss:
        _ok = False
    print(f"  {_s}: " + " · ".join(f"{t} {n:,}" for t, n in _cnts.items())
          + (f"   ⚠️ 누락: {_miss}" if _miss else "   ✓"))
if not _ok:
    print("  ⚠️ 누락 유형이 있습니다. HTP_ROOT 하위에 <split>/<유형>/*.jpg(폴더) "
          "또는 원천/라벨 zip이 있는지 확인하세요.")
print("헬퍼·인덱스 준비 완료.")

In [ ]:
# ── 데이터셋 인벤토리 ─────────────────────────────────────────────
rows = []
for split in HTP_INDEX:
    for htype in HTP_TYPES:
        rows.append({"split": split, "유형": htype,
                     "이미지": len(htp_members(split, htype, "img")),
                     "라벨":  len(htp_members(split, htype, "lbl"))})
inv = pd.DataFrame(rows)
display(inv)
print("총 이미지:", inv["이미지"].sum(), "/ 총 라벨:", inv["라벨"].sum())

In [ ]:
# ── 샘플 이미지 + bbox 시각화 ─────────────────────────────────────
# 한글 폰트는 섹션 1의 setup_korean_font()에서 전역 설정됨 (KOREAN_FONT_NAME)

def show_htp_sample(split, htype, n=2):
    imgs, lbls = htp_members(split, htype, "img"), htp_members(split, htype, "lbl")
    stems = [s for s in sorted(lbls) if s in imgs][:n]
    fig, axes = plt.subplots(1, len(stems), figsize=(6 * len(stems), 6))
    if len(stems) == 1:
        axes = [axes]
    for ax, stem in zip(axes, stems):
        im = Image.open(io.BytesIO(htp_read_bytes(imgs[stem]))).convert("RGB")
        ax.imshow(im); ax.set_title(f"{htype} / {stem}", fontsize=9); ax.axis("off")
        for b in htp_read_json(lbls[stem])["annotations"]["bbox"]:
            ax.add_patch(patches.Rectangle((b["x"], b["y"]), b["w"], b["h"],
                                           fill=False, edgecolor="red", linewidth=1.2))
            ax.text(b["x"], max(b["y"] - 3, 0), b["label"], color="white", fontsize=7,
                    bbox=dict(facecolor="red", alpha=0.6, pad=0.5, edgecolor="none"))
    plt.tight_layout(); plt.show()

show_htp_sample("Training", "집", n=2)
show_htp_sample("Training", "남자사람", n=1)

> **참고 — 사람 그림의 신발 라벨**: 모든 사람 그림에 `운동화` 2개와 `남자구두`/`여자구두` 2개가
> **동시에** 존재한다. 확인 결과 이는 오류가 아니라, 아이가 인물을 그린 뒤 **신발을 따로 확대해서
> 그리는** HTP 검사 절차 때문이다(인물 영역 밖에 신발 그림이 별도로 존재). 중복 제거 불필요.

## 4. 클래스 체계 통합  🏷️

HTP 라벨을 **데이터에서 직접 수집**해 통합 클래스 목록을 만든다(하드코딩 금지).

- 4개 유형에 걸쳐 **총 47개 라벨**. 동일 라벨 문자열은 유형 간 자동 통합된다.
  - `꽃`(집·나무 공통), `나무`(집 그림 배경) → 각 1개 클래스
  - 사람 부위(`눈`·`팔`…)는 남녀 라벨이 동일 → 성별 무관 단일 클래스
  - 성별 구분 라벨만 별도 유지: `남자구두`·`여자구두` (`운동화`는 공통)
- 클래스 id는 **가나다 정렬로 결정론적** 부여 → 재실행 시 항상 동일.

In [ ]:
# ── 전체 라벨 수집 (Training 라벨 전수 스캔, 약 1~2분) ────────────
label_counter = Counter()
label_by_type = defaultdict(Counter)

for htype in HTP_TYPES:
    lbls = htp_members("Training", htype, "lbl")
    for stem, ref in tqdm(lbls.items(), desc=f"scan {htype}", leave=False):
        for b in htp_read_json(ref)["annotations"]["bbox"]:
            label_counter[b["label"]] += 1
            label_by_type[htype][b["label"]] += 1

CLASS_NAMES = sorted(label_counter)
CLASS_TO_ID = {n: i for i, n in enumerate(CLASS_NAMES)}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_ROOT / "classes.json", "w", encoding="utf-8") as f:
    json.dump({"names": CLASS_NAMES, "class_to_id": CLASS_TO_ID}, f, ensure_ascii=False, indent=2)

print(f"통합 클래스 수: {len(CLASS_NAMES)}  (총 bbox {sum(label_counter.values()):,}개)")
rows = [{"id": CLASS_TO_ID[n], "클래스": n, "bbox수(train)": label_counter[n],
         "등장유형": ", ".join(t for t in HTP_TYPES if n in label_by_type[t])}
        for n in CLASS_NAMES]
pd.set_option("display.max_rows", None)
display(pd.DataFrame(rows))

## 5. HTP → YOLO 어노테이션 변환

`{label, x, y, w, h}` (좌상단·픽셀) → `class x_center y_center w h` (정규화).
- 좌표는 이미지 경계로 **클리핑**하고, 폭/높이가 0 이하가 되는 박스는 **버린다**.
- 원본 zip은 **읽기 전용**으로만 접근하고, 결과는 `htp_yolo/stageB_htp/` 에만 생성한다.

In [ ]:
# ── 변환 함수 ─────────────────────────────────────────────────────
def parse_resolution(meta) -> tuple:
    try:
        w, h = str(meta.get("img_resolution", "1280x1280")).lower().split("x")
        return int(w), int(h)
    except Exception:
        return 1280, 1280

def bbox_to_yolo(b, W, H):
    # 좌상단(x,y,w,h) -> 정규화 (cls_id, xc, yc, w, h). 유효하지 않으면 None
    cid = CLASS_TO_ID.get(b["label"])
    if cid is None:
        return None
    x1 = max(0.0, float(b["x"]));            y1 = max(0.0, float(b["y"]))
    x2 = min(float(W), x1 + float(b["w"]));  y2 = min(float(H), y1 + float(b["h"]))
    if x2 - x1 <= 1 or y2 - y1 <= 1:
        return None
    return cid, ((x1+x2)/2)/W, ((y1+y2)/2)/H, (x2-x1)/W, (y2-y1)/H

def select_stems(split, htype, limit):
    imgs, lbls = htp_members(split, htype, "img"), htp_members(split, htype, "lbl")
    common = sorted(set(imgs) & set(lbls))     # 이미지·라벨 둘 다 있는 것만
    random.Random(SEED).shuffle(common)
    return common[:limit] if limit else common

def convert(split, htype, stems, out_dir):
    # 폴더/zip에서 이미지 추출 + YOLO txt 생성 (원본은 읽기 전용, 결과는 stageB_htp에만)
    imgs, lbls = htp_members(split, htype, "img"), htp_members(split, htype, "lbl")
    img_d, lbl_d = out_dir / "images", out_dir / "labels"
    img_d.mkdir(parents=True, exist_ok=True); lbl_d.mkdir(parents=True, exist_ok=True)
    n_box = n_drop = 0
    for stem in tqdm(stems, desc=f"{out_dir.name}/{htype}", leave=False):
        (img_d / f"{stem}.jpg").write_bytes(htp_read_bytes(imgs[stem]))
        d = htp_read_json(lbls[stem])
        W, H = parse_resolution(d["meta"])
        lines = []
        for b in d["annotations"]["bbox"]:
            r = bbox_to_yolo(b, W, H)
            if r is None:
                n_drop += 1; continue
            lines.append("%d %.6f %.6f %.6f %.6f" % r); n_box += 1
        (lbl_d / f"{stem}.txt").write_text("\n".join(lines), encoding="utf-8")
    return n_box, n_drop

print("변환 함수 정의 완료.")

In [ ]:
# ── 실행: 공식 분할 기반 train / val / test 구성 ──────────────────
# Training zip -> train + test(TEST_FRACTION),  Validation zip -> val
if HTP_DS_DIR.exists() and (HTP_DS_DIR / "train" / "images").exists():
    print("기존 변환 결과 재사용:", HTP_DS_DIR.resolve(), "(재생성하려면 폴더 삭제)")
else:
    tot_box = tot_drop = 0
    for htype in HTP_TYPES:
        tr = select_stems("Training", htype, SUBSET_PER_TYPE)
        k = max(1, int(len(tr) * TEST_FRACTION))
        test_stems, train_stems = tr[:k], tr[k:]
        va = select_stems("Validation", htype,
                          max(1, int(SUBSET_PER_TYPE * 0.25)) if SUBSET_PER_TYPE else None)
        for stems, name, split in ((train_stems, "train", "Training"),
                                   (test_stems,  "test",  "Training"),
                                   (va,          "val",   "Validation")):
            b, d = convert(split, htype, stems, HTP_DS_DIR / name)
            tot_box += b; tot_drop += d
    print(f"변환 완료 — 유효 bbox {tot_box:,}개, 제외된 bbox {tot_drop:,}개")

for name in ("train", "val", "test"):
    n = len(list((HTP_DS_DIR / name / "images").glob("*.jpg")))
    print(f"  {name:6s}: {n:,}장")

In [ ]:
# ── 변환 검증: YOLO txt를 되읽어 bbox를 그려 확인 ─────────────────
ID_TO_CLASS = {i: n for n, i in CLASS_TO_ID.items()}
smp = sorted((HTP_DS_DIR / "train" / "images").glob("*.jpg"))[:3]
fig, axes = plt.subplots(1, len(smp), figsize=(6 * len(smp), 6))
if len(smp) == 1: axes = [axes]
for ax, ip in zip(axes, smp):
    im = Image.open(ip); W, H = im.size
    ax.imshow(im); ax.axis("off"); ax.set_title(ip.stem, fontsize=8)
    for line in (ip.parent.parent / "labels" / (ip.stem + ".txt")).read_text().splitlines():
        c, xc, yc, w, h = line.split()
        c = int(c); xc, yc, w, h = float(xc)*W, float(yc)*H, float(w)*W, float(h)*H
        ax.add_patch(patches.Rectangle((xc-w/2, yc-h/2), w, h,
                                       fill=False, edgecolor="lime", linewidth=1.2))
        ax.text(xc-w/2, max(yc-h/2-3, 0), ID_TO_CLASS[c], color="black", fontsize=7,
                bbox=dict(facecolor="lime", alpha=0.7, pad=0.5, edgecolor="none"))
plt.tight_layout(); plt.show()
print("↑ 녹색 박스가 원본 라벨과 일치하면 변환이 올바른 것이다.")

In [ ]:
# ── data.yaml 생성 (Stage B) ──────────────────────────────────────
htp_yaml = HTP_DS_DIR / "data.yaml"
htp_yaml.write_text(
    "path: " + str(HTP_DS_DIR.resolve()) +
    "\ntrain: train/images\nval: val/images\ntest: test/images\n"
    + "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASS_NAMES)),
    encoding="utf-8")
print(htp_yaml.read_text(encoding="utf-8")[:400], "...")
print("\n클래스 수:", len(CLASS_NAMES))

## 6. 파인튜닝 학습 (sketch → HTP)

범용 손그림 모델을 시작 가중치로 HTP 47클래스에 파인튜닝한다.
Stage A와 HTP는 클래스 수가 다르므로 **검출 헤드는 자동 재초기화**되고, 백본/넥의 선화 특징이 전이된다.
(대조군 COCO 경로는 제거 — `baseline.ipynb`에서 이미 열세 확인)

In [ ]:
from ultralytics import YOLO


def train_htp(init_weights, run_name):
    m = YOLO(str(init_weights))
    m.train(data=str(htp_yaml.resolve()),
            epochs=HTP_EPOCHS, imgsz=IMG_SIZE, batch=BATCH, workers=WORKERS,
            device=DEVICE, seed=SEED, patience=PATIENCE,
            project=str(RUNS_DIR), name=run_name, exist_ok=True, verbose=True,
            **HTP_TRAIN_KW)
    return RUNS_DIR / run_name / "weights" / "best.pt"

# 시작 가중치 존재 검사는 Config 셀에서 이미 끝났다.
best_pt = train_htp(SKETCH_BASE, RUN_NAME)

shutil.copy(best_pt, HTP_BEST)   # 최종 산출물로 복사
print("\nHTP 파인튜닝 완료 → 최종 모델:", HTP_BEST.resolve())
print("  시작 가중치:", SKETCH_BASE.name)

## 7. 평가 / 시각화

`test` 분할(공식 Training에서 분리, 학습에 미사용)로 평가한다.
mAP@50 · mAP@50-95 · 클래스별 precision/recall을 출력하고 예측 결과를 시각화한다.

In [ ]:
def evaluate(weights, tag):
    m = YOLO(str(weights))
    r = m.val(data=str(htp_yaml.resolve()), split="test", imgsz=IMG_SIZE,
              batch=BATCH, device=DEVICE, project=str(RUNS_DIR),
              name=f"eval_{tag}", exist_ok=True, verbose=False)
    return {"mAP@50": r.box.map50, "mAP@50-95": r.box.map,
            "precision": r.box.mp, "recall": r.box.mr}, r

metric, raw_result = evaluate(HTP_BEST, "htp")

summary = pd.DataFrame({RUN_NAME: metric}).T
display(summary.style.format("{:.4f}"))

In [ ]:
# ── 클래스별 precision / recall ───────────────────────────────────
r = raw_result
rows = []
for i, c in enumerate(r.box.ap_class_index):
    rows.append({"클래스": CLASS_NAMES[c],
                 "precision": r.box.p[i], "recall": r.box.r[i],
                 "mAP@50": r.box.ap50[i], "mAP@50-95": r.box.ap[i]})
per_cls = pd.DataFrame(rows).sort_values("mAP@50-95", ascending=False)
display(per_cls.style.format({c: "{:.4f}" for c in per_cls.columns if c != "클래스"}))
print("\n성능 하위 10개 클래스:")
display(per_cls.tail(10))

In [ ]:
# ── 실험 기록용 요약 표 (수동 기록 편의) ─────────────────────────
# 이 노트북은 시작 가중치를 고정하고 HTP 파인튜닝만 튜닝하므로,
# 기록 항목도 HTP 파인튜닝 파라미터 + 성능 지표로 한정한다.
# 파라미터를 바꿔 재실행할 때마다 이 셀을 실행 → TSV 한 줄을 스프레드시트에 붙여넣으면 실험 로그가 된다.
import datetime

def _htp_train_sec():
    p = RUNS_DIR / RUN_NAME / "results.csv"
    if not p.exists():
        return None
    return round(float(pd.read_csv(p).iloc[-1]["time"]), 1)

def _worst_classes(r, n=3):
    order = np.argsort(r.box.ap)[:n]
    return "; ".join(f"{CLASS_NAMES[r.box.ap_class_index[i]]}={r.box.ap[i]:.3f}" for i in order)

row = {
    "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "model": HTP_MODEL_NAME,
    "init": SKETCH_MODEL_NAME,      # 실제 사용한 시작 가중치
    "arch": MODEL_ARCH, "imgsz": IMG_SIZE, "batch": BATCH,
    "subset_per_type": SUBSET_PER_TYPE if SUBSET_PER_TYPE else "전체",
    "htp_epochs": HTP_EPOCHS, "patience": PATIENCE,
    "train_kw": str(HTP_TRAIN_KW) if HTP_TRAIN_KW else "-",
    "train_sec": _htp_train_sec(),
    "mAP50": round(metric["mAP@50"], 4),
    "mAP50-95": round(metric["mAP@50-95"], 4),
    "precision": round(metric["precision"], 4),
    "recall": round(metric["recall"], 4),
    "worst3": _worst_classes(raw_result),
}
log_row = pd.DataFrame([row])

print("=== 이번 실험 요약 (sketch_base → HTP) ===")
display(log_row.T.rename(columns={0: "값"}))

print("\n=== 스프레드시트 붙여넣기용 (헤더 행 → 값 행, 탭 구분) ===")
print("\t".join(log_row.columns))
print("\t".join(str(v) for v in log_row.iloc[0]))

In [ ]:
# ── 예측 결과 시각화 (test 분할) ──────────────────────────────────
# ultralytics 기본 plot()은 한글 폰트를 지원하지 않아 라벨이 깨진다.
# → matplotlib(전역 한글 폰트)로 박스를 직접 그린다.

model = YOLO(str(HTP_BEST))
test_imgs = sorted((HTP_DS_DIR / "test" / "images").glob("*.jpg"))[:4]
preds = model.predict([str(p) for p in test_imgs], imgsz=IMG_SIZE, conf=0.25,
                      device=DEVICE, verbose=False)

pred_dir = OUTPUT_ROOT / "predictions"; pred_dir.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, len(preds), figsize=(6 * len(preds), 6))
if len(preds) == 1: axes = [axes]
for ax, p, ip in zip(axes, preds, test_imgs):
    ax.imshow(Image.open(ip).convert("RGB")); ax.axis("off"); ax.set_title(ip.stem, fontsize=8)
    b = p.boxes
    for (x1, y1, x2, y2), cls, conf in zip(b.xyxy.tolist(), b.cls.tolist(), b.conf.tolist()):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       fill=False, edgecolor="red", linewidth=1.2))
        ax.text(x1, max(y1 - 3, 0), f"{CLASS_NAMES[int(cls)]} {conf:.2f}",
                color="white", fontsize=7,
                bbox=dict(facecolor="red", alpha=0.6, pad=0.5, edgecolor="none"))
plt.tight_layout()
fig.savefig(pred_dir / "predictions_overview.png", dpi=120, bbox_inches="tight")
plt.show()
print("예측 이미지 저장:", (pred_dir / "predictions_overview.png").resolve())

## 정리

### 산출물 (모두 `htp_yolo/` 아래)
| 경로 | 내용 |
|---|---|
| `classes.json` | HTP 47개 클래스 ↔ id 매핑 (추론 시 필수) |
| `<HTP_MODEL_NAME>` | **HTP 전용 탐지 모델 (산출물 2)** |
| `stageB_htp/` | 변환된 YOLO 데이터셋 + `data.yaml` |
| `runs/<RUN_NAME>/` | 학습 로그·곡선·`weights/best.pt` |
| `runs/eval_htp/` | test 분할 평가 결과 |
| `predictions/` | 예측 시각화 이미지 |

> 범용 손그림 모델(**산출물 1**)은 [sketch_base.ipynb](sketch_base.ipynb)가 만든다.

### ⚠️ 재학습으로 클래스 집합이 바뀌면
`ai/htp_labels.py`의 47클래스 매핑표도 함께 고쳐야 한다. 불일치는 런타임에
`verify_against_model_names()`가 잡지만, 놓치면 라벨이 전부 `UNKNOWN`으로 접힌다.

### 반복 실험 워크플로
1. Config에서 파라미터 수정 → (필요 시 상단 "캐시 무효화 규칙"대로 폴더 삭제)
2. **커널 재시작 → Run All**
3. 요약표 셀의 TSV 한 줄을 스프레드시트에 추가 기록
   (`runs/<RUN_NAME>/`는 `exist_ok=True`라 재실행 시 덮어써지므로 지표는 매번 기록해 둘 것)

### 본학습 전환 예시 (Config)
```python
IMG_SIZE        = 960
SUBSET_PER_TYPE = None      # 전체 44,800장
HTP_EPOCHS      = 50
HTP_TRAIN_KW    = {"cos_lr": True}
```